# 04 — Prétraitement

Ce notebook explique et justifie les choix de prétraitement de House Prices. Les règles réutilisables sont importées depuis `src/preprocessing/`. Nous partons des anomalies et des distributions observées à l’[étape 3](03_analyse_exploratoire.ipynb).

## 1. Protocole

Dans les CSV, `NA` signale une valeur absente. Le mot `None` est un code valide, notamment pour une maison sans parement de maçonnerie. Nous désactivons donc les autres marqueurs de valeurs manquantes de pandas.

Nous définissons une partition de 80 % des ventes pour l’entraînement et de 20 % pour la validation (`random_state=42`). Pour comparer les choix de nettoyage, nous utilisons une validation croisée à cinq plis sur les 1 460 ventes étiquetées. La comparaison ci-dessous utilise les prédictions hors pli de toutes les ventes. La partition 20 % recoupe ces plis et ne constitue donc pas un test indépendant de ce choix. Tout paramètre appris, dont les médianes et les coefficients du modèle, provient des seules maisons d’entraînement du pli. Le fichier Kaggle `test.csv` sert ici à compter et illustrer les corrections.

In [ ]:
from pathlib import Path
import inspect

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from src.evaluation import rmse_log
from src.preprocessing.steps.cleaning import Cleaner, LARGE_HOUSE_AREA, remove_outliers

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
read_options = {"na_values": ["NA"], "keep_default_na": False}
train = pd.read_csv(root / "data/train.csv", **read_options)
test = pd.read_csv(root / "data/test.csv", **read_options)
X, y = train.drop(columns="SalePrice"), train["SalePrice"]
train_index, holdout_index = train_test_split(X.index, test_size=0.2, random_state=42)
folds = KFold(n_splits=5, shuffle=True, random_state=42)
print(f"Partition 80/20 : {len(train_index)} maisons d’entraînement et {len(holdout_index)} de validation.")

La partition réserve 292 ventes sur 1 460. En validation croisée, chacune des 1 460 maisons contribue une seule prédiction à la comparaison.

## 2. Nettoyage

Nous corrigeons les erreurs manifestes et marquons comme inconnue toute information contradictoire dont la valeur correcte nous échappe. L’imputation relève de la section 3. Une catégorie rare peut encore décrire des maisons différentes. La section 7 évaluera l’intérêt prédictif des variables conservées.

### Doublons et identifiant

Un identifiant unique pourrait masquer des doublons descriptifs. Nous vérifions séparément les identifiants, les lignes complètes et les caractéristiques seules.

In [ ]:
display(pd.Series({
    "Identifiants répétés": train["Id"].duplicated().sum(),
    "Lignes identiques": train.duplicated().sum(),
    "Caractéristiques identiques hors Id et prix": X.drop(columns="Id").duplicated().sum(),
}, name="Répétitions dans train"))
cleaner = Cleaner().fit(X.iloc[train_index])
raw = pd.concat({"train": X, "test": test}, names=["Fichier", "Ligne"])
clean = pd.concat({"train": cleaner.transform(X), "test": cleaner.transform(test)},
                  names=raw.index.names)

def rule_doc(column):
    """Afficher la ligne de documentation correspondant à la règle étudiée."""
    lines = inspect.getdoc(Cleaner.transform).splitlines()
    print("Cleaner.transform —", next(line for line in lines if line.startswith(column + " ")))

def changed(column):
    """Repérer une correction, en considérant deux valeurs manquantes comme identiques."""
    before, after = raw[column], clean[column]
    same = before.eq(after).fillna(False) | (before.isna() & after.isna())
    return ~same

Aucun identifiant ni aucune ligne complète ne se répète dans train. Nous gardons les ventes et retirons `Id` des prédicteurs. Nous le conservons dans les tableaux descriptifs pour retrouver chaque maison.

### 2.1 Type de logement

**Problème.** À l’étape 3 (§ 2), `MSSubClass` est lu comme un entier. Le tableau regroupe toutes les maisons concernées par code et par fichier.

**Décision.** Convertir le code en texte. **Code.** Règle `MSSubClass` de `Cleaner.transform`.

In [ ]:
rule_doc("MSSubClass")
display(pd.crosstab(raw["MSSubClass"], raw.index.get_level_values("Fichier")))

**Justification.** Le code 20 désigne une maison à un étage construite à partir de 1946, le code 60 une maison à deux étages de la même période, et le code 190 une conversion bifamiliale. Un coefficient appliqué au nombre imposerait un ordre et une distance arbitraires. L’encodage catégoriel pourra aussi traiter le code 150, observé seulement dans test.

### 2.2 Revêtements extérieurs

**Problème.** `Exterior1st` indique le revêtement principal des murs et `Exterior2nd` le revêtement secondaire. L’étape 3 (§ 2) relève trois différences d’orthographe entre ces colonnes.

**Décision.** Employer les codes du revêtement principal pour les deux colonnes. **Code.** Règle `Exterior2nd` de `Cleaner.transform`.

In [ ]:
rule_doc("Exterior2nd")
exterior = raw.loc[changed("Exterior2nd"), ["Id", "Exterior1st", "Exterior2nd"]].copy()
exterior["Code harmonisé"] = clean.loc[exterior.index, "Exterior2nd"]
display(exterior.groupby(["Exterior2nd", "Code harmonisé", "Fichier"]).size()
        .unstack("Fichier", fill_value=0))
display(exterior)

**Justification.** Une maison entièrement en bardeaux de bois pouvait porter à la fois `WdShing` et `Wd Shng`. Une comparaison directe aurait alors compté deux matériaux. La correction conserve l’information sur le matériau.

### 2.3 Services publics et variables quasi constantes

**Problème.** La synthèse de l’étape 3 (§ 8) signale sept variables quasi constantes. Nous comptons les modalités de `Utilities`, y compris les valeurs inconnues, puis examinons les exceptions des six autres variables.

**Décision.** Retirer `Utilities` et conserver les six autres caractéristiques. **Code.** Règle `Utilities` de `Cleaner.transform`.

In [ ]:
rule_doc("Utilities")
display(pd.crosstab(raw["Utilities"].fillna("Inconnu"), raw.index.get_level_values("Fichier")))
rare_columns = ["Street", "PoolQC", "Condition2", "RoofMatl", "Heating", "MiscFeature"]
rows = []
for column in rare_columns:
    values = train[column].fillna("Absent")
    majority = values.mode().iat[0]
    unusual = values.ne(majority)
    rows.append([column, majority, unusual.sum(), test[column].fillna("Absent").ne(majority).sum(),
                 y[~unusual].median(), y[unusual].median()])
display(pd.DataFrame(rows, columns=["Variable", "Majorité train", "Exceptions train",
                                     "Exceptions test", "Prix médian majorité ($)",
                                     "Prix médian exceptions ($)"]))

**Justification.** Une seule maison de train a la modalité `NoSeWa`. Dans test, toutes les modalités renseignées sont `AllPub`. Les autres variables ont des exceptions dans les deux fichiers et les prix médians diffèrent dans train. Les petits effectifs limitent l’interprétation de ces écarts. La section 7 examinera leur intérêt pour la prédiction.

### 2.4 Année du garage

**Problème.** Deux garages de test ont une année postérieure à la vente (étape 3, § 4). Le tableau rapproche l’année déclarée du type de garage et des dates de la maison.

**Décision.** Si l’année du garage dépasse celle de la vente de plus d’un an, reprendre l’année de rénovation de cette même maison. **Code.** Règle `GarageYrBlt` de `Cleaner.transform`.

In [ ]:
rule_doc("GarageYrBlt")
garage = raw.loc[raw["GarageYrBlt"].gt(raw["YrSold"]),
                 ["Id", "GarageType", "GarageYrBlt", "YearBuilt", "YearRemodAdd", "YrSold"]].copy()
garage["Année retenue"] = clean.loc[garage.index, "GarageYrBlt"]
display(garage)

**Justification.** Les ventes vont de 2006 à 2010. L’année 2207 du garage attaché de la maison 2593 est une faute de frappe manifeste. Sa rénovation date de 2007, année plausible que la règle retrouve sans consulter l’identifiant. Kaggle exige une prédiction pour chaque maison de test. Garder la maison, l’existence de son garage et une année approximative préserve l’information disponible. Pour la maison 2550, une vente sur plan en 2007 est compatible avec une construction en 2008. La tolérance d’un an conserve cette date.

### 2.5 Parement de maçonnerie

**Problème.** Le parement est une couche décorative de brique ou de pierre sur une partie des murs extérieurs. `MasVnrType` décrit le matériau et `MasVnrArea` la surface en pi². L’étape 3 (§ 4) signale des contradictions entre les deux. Pour situer leur ordre de grandeur, nous calculons la surface médiane des parements de surface positive et de matériau connu dans train.

**Décision.** Corriger la surface bouche-trou. Pour les autres contradictions, marquer le seul champ douteux comme inconnu. **Code.** Règle `MasVnrType` de `Cleaner.transform`.

In [ ]:
rule_doc("MasVnrType")
known = train["MasVnrType"].isin(["BrkFace", "BrkCmn", "Stone"]) & train["MasVnrArea"].gt(0)
print(f"Surface médiane des parements renseignés : {train.loc[known, 'MasVnrArea'].median():g} pi²")
area_changed = changed("MasVnrArea")
type_changed = changed("MasVnrType")
placeholder = area_changed & clean["MasVnrArea"].eq(0)
unknown_area = area_changed & clean["MasVnrArea"].isna()
masonry = raw.loc[type_changed | area_changed, ["Id", "MasVnrType", "MasVnrArea"]].copy()
masonry["Type retenu"] = clean.loc[masonry.index, "MasVnrType"]
masonry["Surface retenue"] = clean.loc[masonry.index, "MasVnrArea"]
display(masonry)

**Justification.** Un parement couvre typiquement environ 200 pi². Une surface d’un pied carré pour un parement déclaré absent est une valeur bouche-trou. À l’inverse, plusieurs centaines de pieds carrés rendent le matériau déclaré absent suspect. Quand le matériau est connu et que la surface vaut zéro, c’est la mesure de surface qui manque.

| Cas | Id dans train | Id dans test | Champ probablement faux et correction |
|---|---|---|---|
| Aucun parement, 1 pi² | 774, 1231 | 2453 | Surface bouche-trou ramenée à 0 |
| Aucun parement, surface supérieure à 1 pi² | 625, 1301, 1335 | 1670 | Type inconnu, surface conservée |
| Brique ou pierre, 0 pi² | 689, 1242 | 2320 | Surface inconnue, matériau conservé |

La section 3 devra distinguer ces inconnues créées par le nettoyage des `NA` d’origine lors de l’imputation.

### 2.6 Valeurs aberrantes

Une vente est aberrante lorsque son prix paraît incohérent avec les caractéristiques de la maison. Un prix élevé peut très bien correspondre à une vaste maison de grande qualité. Nous reprenons les quatre grandes maisons du nuage de points de l’étape 3 (§ 5 et § 7.1).

In [ ]:
display(train.loc[X["GrLivArea"].gt(LARGE_HOUSE_AREA),
                  ["Id", "Neighborhood", "GrLivArea", "OverallQual", "SaleCondition", "SalePrice"]])
print(inspect.getdoc(remove_outliers))

Les maisons 524 et 1299, de qualité 10 à Edwards, se sont vendues moins de 185 000 $ en condition `Partial`. Les deux grandes maisons de NoRidge se sont vendues beaucoup plus cher. Nous comparons le retrait ciblé au retrait par surface seule et au retrait par prix seul.

**Comparaison provisoire.** Un transformateur de pipeline modifie X et doit préserver l’alignement avec y. Nous filtrons donc ensemble X et les prix en dollars **avant** d’ajuster le pipeline, dans la partie entraînement de chaque pli. Toutes les ventes de validation sont prédites.

Les étapes 3 à 7 restent à développer. Pour cette comparaison, nous utilisons uniquement `Cleaner`, une imputation médiane des variables numériques, `log1p` des surfaces (zéro reste défini), une standardisation numérique et un encodage disjonctif des catégories après remplacement provisoire des inconnues par la constante `None`. Les catégories inédites sont ignorées. RidgeCV ajuste une régression ridge sur le logarithme naturel du prix avec 30 pénalisations candidates communes aux quatre options.

In [ ]:
# Les surfaces sont exprimées en pi² dans le dictionnaire d'Ames.
area_columns = [
    "LotArea", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea", "GarageArea", "WoodDeckSF",
    "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "PoolArea",
]

def other_numeric(frame):
    """Les autres grandeurs restent sur leur échelle avant standardisation."""
    return frame.select_dtypes(include="number").columns.difference(area_columns).tolist()

def add_garage_flag(frame):
    """Distinguer la présence du garage pour comparer ses années imputées."""
    return frame.assign(HasGarage=frame["GarageArea"].gt(0).astype(int))

def provisional_model(imputer=None):
    """Reprendre les mêmes transformations et le même modèle dans les deux comparaisons."""
    steps = [Cleaner()]
    area_steps = [FunctionTransformer(np.log1p), StandardScaler()]
    numeric = StandardScaler()
    nominal = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    if imputer is None:
        areas = make_pipeline(SimpleImputer(strategy="median"), *area_steps)
        numeric = make_pipeline(SimpleImputer(strategy="median"), numeric)
        nominal = make_pipeline(SimpleImputer(strategy="constant", fill_value="None"), nominal)
    else:
        steps.extend([imputer, FunctionTransformer(add_garage_flag)])
        areas = make_pipeline(*area_steps)
    columns = ColumnTransformer([
        ("areas", areas, area_columns), ("numeric", numeric, other_numeric),
        ("nominal", nominal, make_column_selector(dtype_exclude="number")),
    ])
    return make_pipeline(*steps, columns, RidgeCV(alphas=np.logspace(-1, 3, 30)))

Le seuil du 1 % pour l’option D est recalculé sur les prix d’entraînement de chaque pli. Pour les scores, nous rassemblons les prédictions hors pli en dollars. Nous calculons ensuite la RMSE des logarithmes sur toutes les maisons, puis sur les 5 % les plus chères d’après les prix observés. Ce dernier seuil sert uniquement à lire les résultats.

In [ ]:
options = ["A · Tout garder", "B · Retrait ciblé", "C · Surface > 4 000 pi²", "D · Prix du centile supérieur"]
scores = []
for option in options:
    predictions = pd.Series(np.nan, index=y.index)
    removed = []
    for fit_index, valid_index in folds.split(X):
        features, prices = X.iloc[fit_index], y.iloc[fit_index]
        if option == options[1]:
            features, prices = remove_outliers(features, prices)
        elif option == options[2]:
            keep = features["GrLivArea"].le(LARGE_HOUSE_AREA)
            features, prices = features.loc[keep], prices.loc[keep]
        elif option == options[3]:
            keep = prices.le(prices.quantile(0.99))
            features, prices = features.loc[keep], prices.loc[keep]
        removed.append(len(fit_index) - len(features))
        model = provisional_model().fit(features, np.log(prices))
        predictions.iloc[valid_index] = np.exp(model.predict(X.iloc[valid_index]))
    assert predictions.notna().all()
    expensive = y.ge(y.quantile(0.95))
    scores.append([option, removed, rmse_log(y, predictions),
                   rmse_log(y[expensive], predictions[expensive])])

comparison = pd.DataFrame(scores, columns=["Option", "Retraits par pli", "RMSE log · toutes",
                                          "RMSE log · 5 % les plus chères"])
display(comparison.round(4))

**Interprétation.** Le retrait du centile supérieur (D) prive le modèle de ventes représentatives du haut de gamme. Sur les 5 % les plus chères, l’erreur passe de 0,1772 (A) à 0,1993 (D). Une maison de test dépasse également 4 000 pi², même si son prix est inconnu. B et C sont proches sur toutes les ventes (0,1283 et 0,1276). B donne la plus faible erreur sur les maisons chères (0,1448 contre 0,1474). Dans le troisième pli, les deux ventes ciblées se trouvent en validation et contribuent toutes deux au score.

**Décision.** Nous retenons B. De Cock conseille pour son jeu pédagogique d’écarter les maisons de plus de 4 000 pi². Il distingue les ventes partielles atypiques des très grandes maisons vendues à un prix cohérent. Sur le train Kaggle, B conserve les deux ventes de NoRidge qui suivent la tendance générale. La section 8 reprendra la comparaison avec la recette complète. [De Cock (2011), *Journal of Statistics Education*, 19(3)](https://doi.org/10.1080/10691898.2011.11889627)

### 2.7 Récapitulatif du nettoyage

Les suppressions de colonnes et la conversion de type concernent chaque ligne. Pour les corrections de valeurs, nous comptons les différences observées entre les données brutes et la sortie de `Cleaner`. La dernière ligne compte le retrait ciblé sur l’ensemble de train à titre descriptif.

In [ ]:
def counts(mask):
    """Compter les maisons concernées dans chacun des deux fichiers."""
    return [int(mask.loc[file].sum()) for file in ("train", "test")]

sizes = [len(train), len(test)]
retained, _ = remove_outliers(X, y)
summary = pd.DataFrame([
    ["Doublons", train.duplicated().sum(), test.duplicated().sum(), "Aucun retrait"],
    ["Id", *sizes, "Retirer la colonne"],
    ["MSSubClass", *sizes, "Convertir en texte"],
    ["Exterior2nd", *counts(changed("Exterior2nd")), "Harmoniser trois codes"],
    ["Utilities", *sizes, "Retirer la colonne"],
    ["GarageYrBlt", *counts(changed("GarageYrBlt")), "Reprendre YearRemodAdd"],
    ["Parement absent, surface minime", *counts(placeholder), "Ramener la surface à 0"],
    ["Parement absent, surface > 1", *counts(type_changed), "Marquer le type inconnu"],
    ["Matériau connu, surface nulle", *counts(unknown_area), "Marquer la surface inconnue"],
    ["Ventes aberrantes (B)", len(X) - len(retained), 0, "Retirer de l’entraînement seulement"],
], columns=["Règle", "Maisons dans train", "Maisons dans test", "Action"])
display(summary)

Le nettoyage conserve toutes les lignes nécessaires aux prédictions Kaggle. Les contradictions de maçonnerie laissent une information exploitable dans l’autre champ. La prochaine étape traitera les valeurs manquantes en tenant compte de cet indice.

## 3. Valeurs manquantes

La section 4 de l’[analyse exploratoire](03_analyse_exploratoire.ipynb) compte les valeurs absentes dans les fichiers bruts. Ici, nous examinons les caractéristiques **après nettoyage**. Un même `NA` peut avoir quatre sens.

| Famille | Ce que signifie la valeur absente | Exemple | Traitement |
|---|---|---|---|
| 1. Équipement absent | La maison ne possède pas l’équipement | Aucun sous-sol | Catégorie `None` et quantités nulles |
| 2. Fausse absence | Une autre colonne atteste la présence | Piscine mesurée, qualité inconnue | Mode ou médiane des maisons équipées |
| 3. Valeur sans objet | La grandeur n’existe pas pour cette maison | Année d’un garage inexistant | Convention numérique fondée sur l’année de la maison |
| 4. Information inconnue | La valeur existe mais manque dans la fiche | Longueur de façade du terrain | Médiane du quartier, règle du dictionnaire ou statistique globale |

### 3.1 Couche 1 — Équipements

Le dictionnaire associe à chaque équipement une mesure, des descriptions et parfois des quantités complémentaires. Le tableau donne les colonnes utilisées par le transformateur. Les modes excluent le code d’absence `None`. Les statistiques d’un équipement proviennent uniquement des maisons d’entraînement dont la mesure est positive.

In [ ]:
from src.preprocessing.steps.missing import EQUIPMENT_GROUPS, MissingValueHandler, consistency_report

display(pd.DataFrame.from_dict(EQUIPMENT_GROUPS, orient="index").rename(columns={
    "measure": "Mesure", "descriptors": "Descriptions", "numbers": "Autres quantités",
}))
before = clean.copy()
handler = MissingValueHandler().fit(before.loc["train"])
after = pd.concat({file: handler.transform(before.loc[file]) for file in ("train", "test")},
                  names=before.index.names)
reports = {file: consistency_report(before.loc[file]) for file in ("train", "test")}
display(pd.concat(reports, names=["Fichier", "Groupe"]))

**Lecture.** Les colonnes R1 à R3 comptent des maisons avant imputation. « Complètes » désigne les équipements de mesure positive dont tous les champs du groupe sont renseignés. Pour `Alley` et `Fence`, le dictionnaire permet d’interpréter directement les NA. Pour `MiscFeature`, un équipement décrit mais de valeur monétaire nulle reste tel quel et se trouve hors de ces quatre colonnes.

- **R1 — Absence cohérente.** Une mesure nulle ou manquante accompagnée de descriptions toutes absentes indique un équipement absent. Pour les sous-sols 2121 et 2189, les quantités manquantes deviennent zéro. Les quantités déjà renseignées sont conservées.
- **R3 — Mesure manquante.** Le garage détaché de la maison 2577 établit son existence. Nous complétons sa surface et sa capacité par les médianes des garages de train.
- **R2 — Fausse absence.** Une mesure positive permet d’imputer les descriptions ou les quantités manquantes. Par exemple, la piscine de la maison 2421 a une surface de 368 pi². Sa qualité reçoit le mode des piscines de train. R3 précède R2 pour que le garage 2577 reçoive aussi ses descriptions.

Les foyers confirment la lecture du dictionnaire dans les deux fichiers (R2 et R3 valent zéro). Dans train, les garages, les piscines et les équipements divers de mesure positive sont également complets. Le tableau suivant détaille chaque champ imputé pour les maisons classées R2 ou R3. La colonne « Origine » distingue les NA bruts des contradictions de parement marquées inconnues à la section 2.

In [ ]:
details = []
for group, fields in EQUIPMENT_GROUPS.items():
    measure = fields["measure"]
    columns = [measure, *fields["descriptors"], *fields["numbers"]]
    missing_fields = before[columns].isna()
    affected = after[measure].gt(0) & missing_fields.any(axis=1)
    for index, row in missing_fields.loc[affected].iterrows():
        rule = "R3" if row[measure] else "R2"
        for column in row.index[row]:
            origin = "CSV" if pd.isna(raw.at[index, column]) else "Nettoyage"
            details.append([index[0], raw.at[index, "Id"], group, rule, column, origin,
                            "NA", after.at[index, column]])
changes = pd.DataFrame(details, columns=["Fichier", "Id", "Groupe", "Règle", "Champ",
                                        "Origine", "Avant", "Après"])
display(changes.sort_values(["Fichier", "Id", "Champ"]).reset_index(drop=True))

La surface manquante d’un parement connu reçoit la médiane des surfaces positives. À l’inverse, l’aire mesurée de la maison 2611 permet de compléter son matériau, même si son NA provient du CSV. Lorsqu’aucun des deux champs n’est renseigné, R1 code l’absence par convention. Les données seules ne permettent pas de confirmer l’existence d’un tel parement.

**Démonstration sur un terrain de basket.** Les colonnes fictives ci-dessous décrivent quatre maisons. Nous ajustons le transformateur sur les deux premières et appliquons ses règles aux quatre. Une seule entrée dans le dictionnaire suffit à décrire cet équipement.

In [ ]:
courts = pd.DataFrame({"CourtArea": [0, 120, 180, np.nan],
                       "CourtQual": [np.nan, "Gd", np.nan, "TA"]},
                      index=["Sans terrain", "Complet", "Qualité inconnue", "Surface inconnue"])
court_group = {"BasketCourt": {"measure": "CourtArea", "descriptors": ["CourtQual"], "numbers": []}}
court_handler = MissingValueHandler(groups=court_group).fit(courts.iloc[:2])
court_result = court_handler.transform(courts)
display(pd.concat({"Avant": courts, "Après": court_result}, axis=1))

La surface de 180 reste renseignée et sa qualité devient `Gd`. La qualité `TA` de la dernière maison est conservée et sa surface reçoit 120, la seule mesure positive de l’échantillon d’ajustement.

### 3.2 Couche 2 — Année du garage, façade et fonctionnalité

**Année du garage.** Pour une maison sans garage, aucune année de construction de garage n’existe. Un zéro créerait un écart artificiel d’environ deux millénaires avec les années observées dans une régression linéaire. Nous mesurons à quelle fréquence maison et garage ont réellement la même année de construction.

In [ ]:
training = before.loc["train"]
known_year = training["GarageArea"].gt(0) & training["GarageYrBlt"].notna()
same_year = training.loc[known_year, "GarageYrBlt"].eq(training.loc[known_year, "YearBuilt"])
print(f"Même année : {same_year.sum()} sur {len(same_year)} garages, soit {same_year.mean():.1%}.")

Dans environ 79 % des cas, les deux dates coïncident. Nous choisissons donc `YearBuilt` comme valeur réaliste pour le modèle. `HasGarage` permettra de distinguer une maison sans garage. La sélection des variables, notamment la stratégie causale, pourra examiner si `GarageYrBlt` apporte encore de l’information au-delà de `YearBuilt`.

**Façade du terrain.** `LotFrontage` mesure en pieds la longueur de la propriété le long de la rue. La médiane du quartier tient compte des différences locales. Le tableau donne des exemples et les deux extrêmes observés dans train.

In [ ]:
frontages = handler.frontage_by_neighborhood_
display(before["LotFrontage"].isna().groupby(level="Fichier").mean().mul(100).round(1)
        .rename("Façades manquantes (%)"))
display(frontages.loc[["BrDale", "OldTown", "Edwards", "NAmes", "NoRidge"]].rename("Médiane (pi)"))
print(f"Étendue des médianes : {frontages.min():g} à {frontages.max():g} pi. "
      f"Médiane globale : {handler.column_fill_['LotFrontage']:g} pi.")

La façade manque pour 17,7 % des ventes de train et 15,6 % des maisons de test. Les médianes des quartiers vont de 21 à 91 pieds, autour d’une médiane globale de 69 pieds. Une façade inconnue dans un quartier inédit ou dépourvu de mesures reçoit cette médiane globale.

**Fonctionnalité.** Le dictionnaire indique « Assume typical unless deductions are warranted ». Une valeur manquante de `Functional` reçoit donc `Typ`. [Dictionnaire des variables](../data/data_description.txt)

**Comparaison des choix.** Nous reprenons le modèle provisoire de la section 2, sur les mêmes cinq plis. L’option B filtre uniquement l’entraînement de chaque pli. Nous remplaçons l’imputation provisoire par `MissingValueHandler` et ajoutons temporairement `HasGarage` après l’imputation dans toutes les variantes. Les transformations des surfaces, l’encodage, la standardisation et RidgeCV restent communs. Pour le garage, la façade est imputée par quartier. Pour la façade, l’année du garage est imputée par `YearBuilt`.

In [ ]:
def imputation_score(garage_option, frontage_option):
    """Calculer une prédiction hors pli pour chacune des ventes étiquetées."""
    predictions = pd.Series(np.nan, index=y.index)
    for fit_index, valid_index in folds.split(X):
        features, prices = remove_outliers(X.iloc[fit_index], y.iloc[fit_index])
        imputer = MissingValueHandler(garage_year_fill=garage_option, lot_frontage_fill=frontage_option)
        model = provisional_model(imputer).fit(features, np.log(prices))
        predictions.iloc[valid_index] = np.exp(model.predict(X.iloc[valid_index]))
    assert predictions.notna().all()
    return rmse_log(y, predictions)

garage_scores = pd.Series({option: imputation_score(option, "neighborhood")
                           for option in ["zero", "year_built", "drop"]}, name="RMSE log")
frontage_scores = pd.Series({"neighborhood": garage_scores["year_built"],
                            "global": imputation_score("year_built", "global")}, name="RMSE log")
display(garage_scores.rename_axis("Option GarageYrBlt").round(4).to_frame())
display(frontage_scores.rename_axis("Option LotFrontage").round(4).to_frame())

**Décision.** Les scores sont proches pour les trois traitements du garage et les deux traitements de la façade. Ces variables et ces imputations ont peu d’influence sur la RMSE dans cette expérience. Nous retenons `year_built` et `neighborhood` pour leur interprétation immobilière. La section 8 pourra vérifier ces choix avec la recette complète.

### 3.3 Couche 3 — Repli sur les statistiques de train

Le filet de sécurité utilise la médiane d’une colonne numérique ou le mode d’une colonne textuelle, même lorsque cette colonne était complète à l’ajustement. Le tableau reprend les caractéristiques complètes dans le CSV d’entraînement mais incomplètes dans le CSV de test. Il donne les **valeurs effectivement imputées** car les couches précédentes traitent déjà certaines inconnues.

In [ ]:
test_only = X.columns[X.notna().all() & test.isna().any()]
rows = []
for column in test_only:
    mask = test[column].isna()
    if column not in after:
        rows.append([column, mask.sum(), "Nettoyage", "Colonne retirée"])
        continue
    layer = "Repli global"
    if any(column in [g["measure"], *g["numbers"], *g["descriptors"]] for g in EQUIPMENT_GROUPS.values()):
        layer = "Équipement"
    elif column == "Functional":
        layer = "Dictionnaire"
    values = after.loc["test"].loc[mask, column].unique()
    rows.append([column, mask.sum(), layer, ", ".join(str(v) for v in values)])
display(pd.DataFrame(rows, columns=["Colonne", "NA dans test", "Règle appliquée", "Valeurs retenues"]))

Le zonage et la qualité de cuisine reçoivent une modalité connue de train. Pour le sous-sol et le garage, la cohérence de l’équipement prime sur le repli global. `Electrical` relève aussi du filet de sécurité, mais son information manquante se trouve dans train.

### 3.4 Vérifications et bilan

Le transformateur utilisé pour les tableaux a été ajusté sur les caractéristiques nettoyées de train uniquement. Nous vérifions les deux fichiers et comparons les corrections de trois maisons à des références recalculées directement sur train.

In [ ]:
missing_counts = pd.DataFrame({
    "Avant": before.isna().groupby(level="Fichier").sum().sum(axis=1),
    "Après": after.isna().groupby(level="Fichier").sum().sum(axis=1),
})
display(missing_counts)
assert missing_counts["Après"].eq(0).all()
assert before.index.equals(after.index) and before.columns.equals(after.columns)
houses = after.loc["test"].set_axis(test["Id"])
pool_mode = training.loc[training["PoolArea"].gt(0), "PoolQC"].dropna().mode().iat[0]
garage_median = training.loc[training["GarageArea"].gt(0), "GarageArea"].median()
assert houses.at[2421, "PoolQC"] == pool_mode
assert houses.at[2577, "GarageArea"] == garage_median
assert houses.at[2577, "GarageYrBlt"] == houses.at[2577, "YearBuilt"]
basement_areas = ["TotalBsmtSF", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF"]
assert houses.loc[2121, basement_areas].eq(0).all()

Chaque fichier conserve ses maisons et ses colonnes. Les contrôles vérifient la disparition des NA et le sens des valeurs produites. Le bilan compte une maison une seule fois par ligne lorsqu’un de ses champs a été rempli. Une même maison peut figurer dans plusieurs lignes.

In [ ]:
from src.preprocessing.steps.missing import NO_MEASURE_ABSENCE

summary_rows = []
equipment_columns = []
absence = pd.Series(False, index=before.index)
unknown = absence.copy()
for group in EQUIPMENT_GROUPS.values():
    measure = group["measure"]
    columns = [measure, *group["descriptors"], *group["numbers"]]
    equipment_columns.extend(columns)
    affected = before[columns].isna().any(axis=1)
    absence |= affected & after[measure].eq(0)
    unknown |= affected & after[measure].gt(0)

def summarize(family, columns, rule, mask):
    """Compter les maisons touchées dans chaque fichier."""
    summary_rows.append([family, columns, rule, *counts(mask)])

summarize("1 · Absence", "Six groupes", "R1 · None ou zéro", absence)
summarize("1 · Absence", "Alley, Fence", "None", before[NO_MEASURE_ABSENCE].isna().any(axis=1))
summarize("2 · Fausse absence", "Six groupes", "R2/R3 · mode ou médiane des équipées", unknown)
year_missing = before["GarageYrBlt"].isna()
summarize("3 · Sans objet", "GarageYrBlt sans garage", "YearBuilt", year_missing & after["GarageArea"].eq(0))
summarize("4 · Inconnue", "GarageYrBlt avec garage", "YearBuilt", year_missing & after["GarageArea"].gt(0))
summarize("4 · Inconnue", "LotFrontage", "Médiane du quartier ou globale", before["LotFrontage"].isna())
summarize("4 · Inconnue", "Functional", "Typ", before["Functional"].isna())
special = [*equipment_columns, *NO_MEASURE_ABSENCE, "GarageYrBlt", "LotFrontage", "Functional"]
remaining = before.drop(columns=special)
fallback_cols = remaining.columns[remaining.isna().any()]
summarize("4 · Inconnue", ", ".join(fallback_cols), "Mode ou médiane globale",
          remaining.isna().any(axis=1))
display(pd.DataFrame(summary_rows, columns=["Famille", "Colonnes", "Règle", "Maisons train", "Maisons test"]))

**Limite.** Une façade imputée de 69 pieds devient indiscernable d’une mesure observée de 69 pieds. Par simplicité, nous gardons la valeur imputée seule. Une expérience ultérieure pourrait mesurer l’apport d’un indicateur signalant que la mesure manquait à l’origine.

## 4. Création de caractéristiques

À compléter.

## 5. Encodage des variables catégorielles

À compléter.

## 6. Normalisation et standardisation

À compléter.

## 7. Sélection des variables (stratégies)

À compléter.

## 8. Pipelines et reproductibilité

À compléter.

## 9. Bilan

À compléter.